# Metadata-Only Join Graph Inference: Reference Pipeline

This notebook is the reference implementation of the pipeline described in the paper.
It infers typed join relationships (PK–FK and PK–PK) from **schema metadata only**:
attribute names, table names, datatypes, and declared key status. No data values and no
declared foreign keys are used as input. Every candidate check is an LLM call, so the
pipeline treats verification as a budget and prunes as it goes.

**Stages** (matching the paper):

1. **Candidate pair pruning**: attribute pruning (datatype + LLM attribute role), then attribute-pair pruning.
2. **PK–PK inference**: verify key-key pairs, build logical key groups, prune table pairs inside a group.
3. **CAPRI (PK–FK inference)**: rank cluster-pair regions, verify candidates in ranked batches, expand a region only while it yields acceptances, and prune each accepted FK's competing targets immediately.
4. **Matrix completion + refinement**: low-rank completion over still-unknown pairs; LLM-verify the survivors.
5. **FK-guided PK–FK recovery**: assign unmatched FK-like attributes by comparing them to already-accepted FK members of a PK group.
6. **Matrix finalization + evaluation**.

**Datasets.** Set `DATASET` to one of `tpch`, `tpcds`, `job`, `bird` in the configuration cell.
For BIRD-SQL, the pipeline runs once per database schema (`BIRD_DB_ID`).

**LLM backend.** With `LLM_BACKEND = "manual"`, each stage writes its verification requests to
`<output>/llm_requests/<task>.json` and reads answers from `<output>/llm_results/<task>.json`;
re-run the notebook after the results are available. With `LLM_BACKEND = "openai"`, requests are sent directly.
Every verified item is counted in `LLM_CALL_LOG`, which is how verifier calls are reported.

# 0. Configuration

In [ ]:
from pathlib import Path
import os, re, json, math
from collections import Counter, defaultdict, deque
from typing import Dict, List, Tuple, Set, Optional

import numpy as np
import pandas as pd

# ---------------- Dataset selection ----------------
DATASET = "tpch"          # "tpch" | "tpcds" | "job" | "bird"
BIRD_DB_ID = "california_schools"   # used only when DATASET == "bird"

DATASET_CONFIG = {
    "tpch":  {"schema_path": "data/tpch/schema.sql",  "workload_dir": None},
    "tpcds": {"schema_path": "data/tpcds/schema.sql", "workload_dir": None},
    "job":   {"schema_path": "data/job/schema.sql",   "workload_dir": "data/job/queries"},
    "bird":  {"schema_path": "data/bird/{db}/schema.sql", "workload_dir": "data/bird/{db}/queries"},
}

cfg = DATASET_CONFIG[DATASET]
SCHEMA_PATH = cfg["schema_path"].format(db=BIRD_DB_ID)
WORKLOAD_DIR = cfg["workload_dir"].format(db=BIRD_DB_ID) if cfg["workload_dir"] else None
RUN_NAME = DATASET if DATASET != "bird" else f"bird_{BIRD_DB_ID}"
OUTPUT_DIR = os.path.join("outputs", RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- Thresholds ----------------
TAU_SIM_PKPK      = 0.70   # embedding similarity gate for PK-PK candidates
TAU_LLM           = 0.80   # LLM acceptance threshold (PK-PK, refinement, finalization)
TAU_LLM_CAPRI     = 0.90   # LLM acceptance threshold inside CAPRI batches
CAPRI_BATCH_RATIO = 0.20   # batch size = ceil(ratio * region size)
TABLE_CLUSTER_DIST = 0.35  # agglomerative clustering distance (tables)
ATTR_CLUSTER_DIST  = 0.20  # agglomerative clustering distance (key attributes)
COMPLETION_RANK = 5
COMPLETION_MAX_ITER = 100
COMPLETION_TOL = 1e-5
TAU_PROP = 0.20            # completion score below which an unknown pair is pruned
FKFK_TOP_K_PER_GROUP = 2
TAU_SIM_FKFK = 0.70
TAU_LLM_FKFK = 0.90
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------- LLM backend ----------------
LLM_BACKEND = "manual"     # "manual" | "openai"
LLM_MODEL = "gpt-4o"

## Dataset-specific name expansions

Short prefixes and abbreviations hurt embedding similarity. Table prefixes (e.g. `ss_`, `l_`) are
detected automatically per table; the dictionaries below only expand abbreviations.

In [ ]:
TOKEN_EXPANSIONS = {
    "tpch": {
        "suppkey": "supplier key", "partkey": "part key", "custkey": "customer key",
        "orderkey": "order key", "nationkey": "nation key", "regionkey": "region key",
        "linenumber": "line number",
    },
    "tpcds": {
        "sk": "surrogate key", "addr": "address", "demo": "demographics", "cdemo": "customer demographics",
        "hdemo": "household demographics", "promo": "promotion", "cust": "customer",
        "bill": "billing", "ship": "ship", "sold": "sold", "returned": "returned",
    },
    "job": {"info": "information", "id": "identifier", "imdb": "imdb"},
    "bird": {"id": "identifier", "num": "number", "cd": "code", "no": "number"},
}[DATASET]

GENERIC_KEY_SUFFIXES = {"id", "ids", "key", "keys", "sk", "pk", "fk", "uuid", "guid",
                        "code", "number", "num", "no", "identifier", "surrogate"}
ROLE_TOKENS = {"bill", "billing", "ship", "shipping", "sold", "sale", "sales", "return",
               "returned", "current", "prior", "previous", "start", "end", "from", "to"}

## LLM verification interface

In [ ]:
LLM_REQ_DIR = os.path.join(OUTPUT_DIR, "llm_requests")
LLM_RES_DIR = os.path.join(OUTPUT_DIR, "llm_results")
os.makedirs(LLM_REQ_DIR, exist_ok=True)
os.makedirs(LLM_RES_DIR, exist_ok=True)

LLM_CALL_LOG: List[dict] = []   # one entry per verified item, used for call/cost accounting


def _load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read().strip()
    return json.loads(raw) if raw else default


def _call_openai(prompt: str) -> dict:
    """Single verification call. Expects the model to answer with a JSON object."""
    from openai import OpenAI
    client = OpenAI()   # reads OPENAI_API_KEY from the environment
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    return json.loads(resp.choices[0].message.content)


def llm_verify(task: str, items: List[dict]) -> Dict[str, dict]:
    """
    Verify a list of items. Each item has a unique "item_id" and a "prompt".
    Returns {item_id: answer} for every item that has an answer.

    manual: requests are appended to llm_requests/<task>.json; answers are read from
            llm_results/<task>.json (a list of objects with "item_id").
    openai: requests are sent directly and cached in llm_results/<task>.json.
    """
    res_path = os.path.join(LLM_RES_DIR, f"{task}.json")
    cached = {r["item_id"]: r for r in _load_json(res_path, [])}
    answers, missing = {}, []

    for it in items:
        if it["item_id"] in cached:
            answers[it["item_id"]] = cached[it["item_id"]]
        else:
            missing.append(it)

    if missing and LLM_BACKEND == "openai":
        for it in missing:
            out = _call_openai(it["prompt"])
            out["item_id"] = it["item_id"]
            cached[it["item_id"]] = out
            answers[it["item_id"]] = out
        with open(res_path, "w", encoding="utf-8") as f:
            json.dump(list(cached.values()), f, indent=2)
        missing = []

    if missing:   # manual mode: export for offline answering
        req_path = os.path.join(LLM_REQ_DIR, f"{task}.json")
        existing = {r["item_id"]: r for r in _load_json(req_path, [])}
        for it in missing:
            existing[it["item_id"]] = it
        with open(req_path, "w", encoding="utf-8") as f:
            json.dump(list(existing.values()), f, indent=2)
        print(f"[{task}] {len(missing)} items exported for LLM verification -> {req_path}")

    for item_id in answers:
        LLM_CALL_LOG.append({"task": task, "item_id": item_id})
    return answers


def parse_decision(ans: dict, threshold: float) -> Tuple[bool, float]:
    """Normalize an LLM answer into (accepted, probability)."""
    prob = float(ans.get("probability", ans.get("score", 0.0)))
    prob = max(0.0, min(1.0, prob))
    dec = str(ans.get("decision", "")).strip().upper()
    if dec in {"NOT_JOIN", "NO", "FALSE", "INVALID", "REJECT", "NOT_SAME_GROUP", "NOT_KEY"}:
        return False, prob
    return prob >= threshold, prob

# 1. Load and parse schema

In [ ]:
def strip_sql_comments(sql: str) -> str:
    sql = re.sub(r"/\*.*?\*/", "", sql, flags=re.DOTALL)
    return re.sub(r"--.*?$", "", sql, flags=re.MULTILINE)


def norm_id(name: str) -> str:
    name = str(name).strip().strip('`"[]')
    if "." in name:
        name = name.split(".")[-1]
    return name.strip('`"[]').lower()


def split_top_level_commas(body: str) -> List[str]:
    parts, cur, depth = [], [], 0
    for ch in body:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append("".join(cur).strip()); cur = []
        else:
            cur.append(ch)
    if "".join(cur).strip():
        parts.append("".join(cur).strip())
    return parts


def extract_create_table_blocks(sql: str) -> List[Tuple[str, str]]:
    sql = strip_sql_comments(sql)
    pat = re.compile(r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?([^\s(]+)\s*\(", re.IGNORECASE)
    blocks = []
    for m in pat.finditer(sql):
        j, depth = m.end(), 1
        while j < len(sql) and depth > 0:
            depth += {"(": 1, ")": -1}.get(sql[j], 0)
            j += 1
        blocks.append((m.group(1), sql[m.end():j - 1]))
    return blocks


TYPE_STOP = {"NOT", "NULL", "PRIMARY", "UNIQUE", "DEFAULT", "REFERENCES", "CHECK",
             "CONSTRAINT", "COLLATE", "AUTO_INCREMENT", "AUTOINCREMENT"}

def extract_type(rest: str) -> str:
    toks = []
    for t in re.findall(r"[A-Za-z_]+(?:\s*\([^)]*\))?", rest):
        if t.split("(")[0].strip().upper() in TYPE_STOP:
            break
        toks.append(t.replace(" ", ""))
    return " ".join(toks).upper() or "UNKNOWN"


def parse_schema(sql_text: str) -> Dict[str, dict]:
    """
    Returns {table: {"columns": {col: dtype}, "pk": [...], "unique": [[...]], "fks": [...]}}.
    Foreign keys are parsed ONLY to build ground truth; they are never pipeline input.
    """
    tables = {}
    for raw_table, body in extract_create_table_blocks(sql_text):
        t = norm_id(raw_table)
        info = {"columns": {}, "pk": [], "unique": [], "fks": []}
        for d in split_top_level_commas(body):
            up = d.upper().strip()
            m = re.search(r"PRIMARY\s+KEY\s*\((.*?)\)", d, re.I | re.S)
            if up.startswith(("PRIMARY", "CONSTRAINT")) and m:
                info["pk"] = [norm_id(x) for x in m.group(1).split(",")]; continue
            m = re.search(r"FOREIGN\s+KEY\s*\((.*?)\)\s*REFERENCES\s+([^\s(]+)\s*\((.*?)\)", d, re.I | re.S)
            if m:
                info["fks"].append(([norm_id(x) for x in m.group(1).split(",")], norm_id(m.group(2)),
                                    [norm_id(x) for x in m.group(3).split(",")])); continue
            m = re.match(r"(?:CONSTRAINT\s+\S+\s+)?UNIQUE\s*\((.*?)\)", d.strip(), re.I | re.S)
            if m:
                info["unique"].append([norm_id(x) for x in m.group(1).split(",")]); continue
            if up.startswith(("CHECK", "KEY ", "INDEX")):
                continue
            parts = d.split(None, 1)
            if len(parts) < 2:
                continue
            col, rest = norm_id(parts[0]), parts[1]
            info["columns"][col] = extract_type(rest)
            if re.search(r"\bPRIMARY\s+KEY\b", rest, re.I):
                info["pk"] = [col]
            if re.search(r"\bUNIQUE\b", rest, re.I):
                info["unique"].append([col])
            m = re.search(r"REFERENCES\s+([^\s(]+)\s*\(([^)]+)\)", rest, re.I)
            if m:
                info["fks"].append(([col], norm_id(m.group(1)), [norm_id(m.group(2))]))
        tables[t] = info
    return tables


with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    sql_text = f.read()

tables = parse_schema(sql_text)

attr_names = sorted(f"{t}.{c}" for t, info in tables.items() for c in info["columns"])
dtype_map = {f"{t}.{c}": d for t, info in tables.items() for c, d in info["columns"].items()}
primary_key_attrs = {f"{t}.{c}" for t, info in tables.items() for c in info["pk"]}
unique_attrs = {f"{t}.{c}" for t, info in tables.items() for u in info["unique"] if len(u) == 1 for c in u}
key_attrs = primary_key_attrs | unique_attrs     # "declared key status"

def table_of(a: str) -> str: return a.split(".", 1)[0]
def col_of(a: str) -> str: return a.split(".", 1)[1]
def norm_pair(a, b) -> Tuple[str, str]: return tuple(sorted((str(a), str(b))))

print(f"{RUN_NAME}: {len(tables)} tables, {len(attr_names)} attributes, {len(key_attrs)} declared keys")

# 2. Ground truth (evaluation only)

In [ ]:
def schema_fk_pairs(tables) -> Set[Tuple[str, str]]:
    pairs = set()
    for t, info in tables.items():
        for lcols, rt, rcols in info["fks"]:
            for lc, rc in zip(lcols, rcols):
                pairs.add(norm_pair(f"{t}.{lc}", f"{rt}.{rc}"))
    for m in re.finditer(r"ALTER\s+TABLE\s+([^\s;]+).*?FOREIGN\s+KEY\s*\((.*?)\)\s*REFERENCES\s+([^\s(;]+)\s*\((.*?)\)",
                         strip_sql_comments(sql_text), re.I | re.S):
        for lc, rc in zip(m.group(2).split(","), m.group(4).split(",")):
            pairs.add(norm_pair(f"{norm_id(m.group(1))}.{norm_id(lc)}", f"{norm_id(m.group(3))}.{norm_id(rc)}"))
    return pairs


def workload_join_pairs(folder: Optional[str]) -> Set[Tuple[str, str]]:
    """Equi-join predicates a.x = b.y from a SQL workload, with aliases resolved."""
    if not folder or not Path(folder).is_dir():
        return set()
    pairs = set()
    for p in sorted(Path(folder).glob("*.sql")):
        q = re.sub(r"\s+", " ", strip_sql_comments(p.read_text(encoding="utf-8")))
        alias = {}
        for t, a in re.findall(r"\b(?:FROM|JOIN|,)\s+([A-Za-z_]\w*)(?:\s+(?:AS\s+)?([A-Za-z_]\w*))?", q, re.I):
            if a and a.upper() not in {"ON", "WHERE", "JOIN", "INNER", "LEFT", "RIGHT", "GROUP", "ORDER"}:
                alias[a.lower()] = t.lower()
            alias.setdefault(t.lower(), t.lower())
        for la, lc, ra, rc in re.findall(r"(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)", q):
            a = f"{alias.get(la.lower(), la.lower())}.{lc.lower()}"
            b = f"{alias.get(ra.lower(), ra.lower())}.{rc.lower()}"
            if a in dtype_map and b in dtype_map and table_of(a) != table_of(b):
                pairs.add(norm_pair(a, b))
    return pairs


gt_set = schema_fk_pairs(tables) | workload_join_pairs(WORKLOAD_DIR)
print("Ground-truth join pairs:", len(gt_set))

# Stage 1. Candidate pair pruning

**Attribute pruning.** Drop attributes whose datatype cannot support a key (decimal measures,
free text), then ask the LLM whether each remaining attribute is semantically key-like.

**Attribute-pair pruning.** Keep a pair only if the two attributes are in different tables,
both are key-like, their datatype families are compatible, and at least one side is a declared key.

The candidate matrix `M` uses `-1` = unknown, `0` = pruned/rejected, `>0` = accepted score.

In [ ]:
def dtype_family(d: str) -> str:
    d = str(d).lower()
    if any(x in d for x in ["text", "clob", "blob", "json", "xml"]): return "free_text"
    if any(x in d for x in ["decimal", "numeric", "float", "double", "real", "money"]): return "measure"
    if "int" in d or "serial" in d: return "integer"
    if any(x in d for x in ["char", "string"]): return "string"
    if any(x in d for x in ["date", "time"]): return "datetime"
    if "bool" in d: return "boolean"
    return "other"

NON_KEY_FAMILIES = {"measure", "free_text", "boolean"}

def dtypes_compatible(a: str, b: str) -> bool:
    fa, fb = dtype_family(dtype_map[a]), dtype_family(dtype_map[b])
    return fa == fb and fa not in NON_KEY_FAMILIES

# --- 1A. Datatype filter ---
dtype_survivors = [a for a in attr_names if dtype_family(dtype_map[a]) not in NON_KEY_FAMILIES]

# --- 1B. LLM attribute-role check ---
def attribute_role_prompt(a: str) -> str:
    return (
        "You are given one database column from a schema without data values.\n"
        f"Table: {table_of(a)}\nColumn: {col_of(a)}\nDatatype: {dtype_map[a]}\n"
        f"Declared key: {a in key_attrs}\n"
        "Decide whether this column is key-like, i.e. could participate in a join as a primary or "
        "foreign key. Descriptions, comments, names, prices, quantities, flags and free text are not "
        "keys unless they clearly act as identifiers.\n"
        'Answer as JSON: {"decision": "POSSIBLE_KEY" or "NOT_KEY", "probability": 0-1, "reason": "..."}'
    )

role_items = [{"item_id": a, "attribute": a, "prompt": attribute_role_prompt(a)} for a in dtype_survivors]
role_answers = llm_verify("attribute_role", role_items)

possible_key_attrs = set(key_attrs)   # declared keys are always kept
for a, ans in role_answers.items():
    if str(ans.get("decision", "")).upper() in {"POSSIBLE_KEY", "POSSIBLE_IDENTIFIER", "PRIMARY_KEY", "FOREIGN_KEY"}:
        possible_key_attrs.add(a)
not_key_attrs = set(attr_names) - possible_key_attrs

# --- 1C. Attribute-pair pruning ---
M = pd.DataFrame(-1.0, index=attr_names, columns=attr_names)
for a in attr_names:
    M.loc[a, a] = 0.0

def set_pair(M_df, a, b, v):
    M_df.loc[a, b] = v; M_df.loc[b, a] = v

def keep_pair(a: str, b: str) -> bool:
    return (table_of(a) != table_of(b)
            and a not in not_key_attrs and b not in not_key_attrs
            and dtypes_compatible(a, b)
            and (a in key_attrs or b in key_attrs))

for i, a in enumerate(attr_names):
    for b in attr_names[i + 1:]:
        if not keep_pair(a, b):
            set_pair(M, a, b, 0.0)

def unknown_pairs(M_df) -> List[Tuple[str, str]]:
    idx = list(M_df.index); vals = M_df.values
    return [(idx[i], idx[j]) for i, j in zip(*np.where(np.triu(vals == -1, k=1)))]

STAGE_CANDIDATES = {"after_pair_pruning": len(unknown_pairs(M))}
print("Possible-key attributes:", len(possible_key_attrs))
print("Candidate pairs after pruning:", STAGE_CANDIDATES["after_pair_pruning"])

## Name normalization and embeddings

Used by PK–PK gating, clustering, CAPRI ranking, and FK-guided recovery. Each attribute is reduced
to its *entity words*: the table prefix, generic key suffixes, and role words are removed, and
abbreviations are expanded (e.g. `cs_bill_customer_sk` → `customer`).

In [ ]:
def tokens(s: str) -> List[str]:
    return [t for t in re.split(r"[_\W]+", str(s).lower()) if t]

def singular(t: str) -> str:
    if len(t) > 4 and t.endswith("ies"): return t[:-3] + "y"
    if len(t) > 3 and t.endswith("s") and not t.endswith("ss"): return t[:-1]
    return t

# Detect each table's column prefix automatically (e.g. "ss" for store_sales, "l" for lineitem).
table_prefix = {}
for t, info in tables.items():
    firsts = Counter(tokens(c)[0] for c in info["columns"] if len(tokens(c)) > 1)
    if firsts:
        tok, n = firsts.most_common(1)[0]
        if len(tok) <= 4 and n >= 0.6 * len(info["columns"]):
            table_prefix[t] = tok

def expand(tok: str) -> List[str]:
    return TOKEN_EXPANSIONS.get(tok, tok).split()

def entity_words(a: str) -> str:
    out = []
    for tok in tokens(col_of(a)):
        if tok == table_prefix.get(table_of(a)):
            continue
        tok = singular(tok)
        if tok in GENERIC_KEY_SUFFIXES or tok in ROLE_TOKENS:
            continue
        out.extend(w for w in expand(tok) if w not in GENERIC_KEY_SUFFIXES)
    if not out:   # e.g. JOB "id": fall back to the table name
        out = [singular(w) for w in tokens(table_of(a))]
    return " ".join(out)

def attr_text(a: str) -> str:
    return (f"table meaning: {' '.join(expand(' '.join(tokens(table_of(a)))))}. "
            f"key entity meaning: {entity_words(a)}.")

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
except Exception:
    embedding_model = None

def embed(texts: List[str]) -> np.ndarray:
    if embedding_model is not None:
        return np.asarray(embedding_model.encode(texts, normalize_embeddings=True, show_progress_bar=False))
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    return normalize(TfidfVectorizer(ngram_range=(1, 2)).fit_transform(texts)).toarray()

key_like = sorted(possible_key_attrs)
_E_entity = dict(zip(key_like, embed([entity_words(a) for a in key_like])))
_E_attr = dict(zip(key_like, embed([attr_text(a) for a in key_like])))

def sim_entity(a, b) -> float:   # short "entity word" similarity (PK-PK gate)
    return float(np.dot(_E_entity[a], _E_entity[b])) if a in _E_entity and b in _E_entity else 0.0

def sim(a, b) -> float:          # attribute similarity used for ranking
    return float(np.dot(_E_attr[a], _E_attr[b])) if a in _E_attr and b in _E_attr else 0.0

# Stage 2. PK–PK inference

Key-key pairs are gated by entity-word similarity (`sim ≥ τ_sim`) and verified by the LLM.
Pairs below the gate are **soft-skipped**: they stay unknown and can still be recovered by matrix
completion. Accepted PK–PK pairs form *logical key groups* (connected components over attributes).
Two tables that share a logical group are already connected, so their remaining unknown pairs are
pruned (**PK–PK table-pair pruning**).

In [ ]:
def pkpk_prompt(a, b) -> str:
    return (
        "Both columns are declared keys (or part of a composite key) in a schema without data values.\n"
        f"Column A: {a} ({dtype_map[a]})\nColumn B: {b} ({dtype_map[b]})\n"
        "Decide whether A and B identify the same entity, so that A = B is a valid join. This can "
        "happen when one side is part of a composite key that behaves like a foreign key to the other.\n"
        'Answer as JSON: {"decision": "JOIN" or "NOT_JOIN", "probability": 0-1, "reason": "..."}'
    )

pkpk_candidates, soft_skipped_pkpk = [], set()
for a, b in unknown_pairs(M):
    if a in key_attrs and b in key_attrs:
        if sim_entity(a, b) >= TAU_SIM_PKPK:
            pkpk_candidates.append((a, b))
        else:
            soft_skipped_pkpk.add(norm_pair(a, b))

pkpk_answers = llm_verify("pkpk", [
    {"item_id": f"{a}|{b}", "left_column": a, "right_column": b, "prompt": pkpk_prompt(a, b)}
    for a, b in pkpk_candidates
])

accepted_pkpk = set()
for a, b in pkpk_candidates:
    ans = pkpk_answers.get(f"{a}|{b}")
    if ans is None:
        continue
    ok, p = parse_decision(ans, TAU_LLM)
    set_pair(M, a, b, p if ok else 0.0)
    if ok:
        accepted_pkpk.add(norm_pair(a, b))

# --- Logical key groups (attribute-level connected components; no transitive table merge) ---
graph = defaultdict(set)
for a, b in accepted_pkpk:
    graph[a].add(b); graph[b].add(a)
logical_groups, seen = [], set()
for node in graph:
    if node in seen:
        continue
    comp, q = set(), deque([node]); seen.add(node)
    while q:
        cur = q.popleft(); comp.add(cur)
        for nxt in graph[cur] - seen:
            seen.add(nxt); q.append(nxt)
    logical_groups.append(comp)

table_pair_groups = defaultdict(set)
for gid, comp in enumerate(logical_groups):
    tabs = sorted({table_of(a) for a in comp})
    for i, t1 in enumerate(tabs):
        for t2 in tabs[i + 1:]:
            table_pair_groups[norm_pair(t1, t2)].add(gid)

def share_logical_group(a, b) -> bool:
    return table_of(a) == table_of(b) or bool(table_pair_groups.get(norm_pair(table_of(a), table_of(b))))

# --- PK-PK table-pair pruning (soft-skipped PK-PK pairs are protected) ---
for a, b in unknown_pairs(M):
    if share_logical_group(a, b) and norm_pair(a, b) not in soft_skipped_pkpk:
        set_pair(M, a, b, 0.0)

STAGE_CANDIDATES["after_pkpk"] = len(unknown_pairs(M))
print("PK-PK verified:", len(pkpk_candidates), "| accepted:", len(accepted_pkpk),
      "| soft-skipped:", len(soft_skipped_pkpk), "| logical groups:", len(logical_groups))
print("Unknown pairs after PK-PK stage:", STAGE_CANDIDATES["after_pkpk"])

# Stage 3. CAPRI: PK–FK inference

1. **Cluster-pair ranking.** Tables are clustered semantically; each PK–FK candidate falls in a
   cluster-pair region `R`. Regions are visited in descending order of
   `π(R) = 0.6 · max sim(candidates in R) + 0.4 · max table similarity in R`.
2. **Candidate ranking.** Inside a region, candidates are ranked by `sim`.
3. **Batch verification.** The next ranked batch (`ceil(ratio · |R|)`) is sent to the LLM.
4. **FK-target conflict pruning.** When a candidate is accepted, every other unknown target for
   that FK is pruned immediately, before it is ever verified.
5. **Expand / terminate.** If the batch had at least one acceptance, verify the next batch;
   otherwise stop and move to the next region. Unvisited candidates stay unknown (skipping is not a
   negative decision) and are handed to matrix completion.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def cluster(emb: np.ndarray, labels: List[str], dist: float) -> Dict[str, int]:
    if len(labels) <= 1:
        return {l: 0 for l in labels}
    try:
        model = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average", distance_threshold=dist)
    except TypeError:
        model = AgglomerativeClustering(n_clusters=None, affinity="cosine", linkage="average", distance_threshold=dist)
    return dict(zip(labels, model.fit_predict(emb)))

table_keys = defaultdict(list)
for a in key_like:
    table_keys[table_of(a)].append(a)
table_names = sorted(table_keys)
table_text = [f"table meaning: {' '.join(tokens(t))}. key entities: "
              f"{', '.join(sorted({entity_words(a) for a in table_keys[t]}))}." for t in table_names]
E_table = dict(zip(table_names, embed(table_text)))
table_cluster = cluster(np.stack([E_table[t] for t in table_names]), table_names, TABLE_CLUSTER_DIST)

def table_sim(t1, t2) -> float:
    return float(np.dot(E_table[t1], E_table[t2])) if t1 in E_table and t2 in E_table else 0.0

def orient(a, b) -> Optional[Tuple[str, str]]:
    """(pk, fk) when exactly one side is a declared key."""
    if (a in key_attrs) ^ (b in key_attrs):
        return (a, b) if a in key_attrs else (b, a)
    return None

# --- Candidate pool ---
cands = []
for a, b in unknown_pairs(M):
    o = orient(a, b)
    if o is None:
        continue
    pk, fk = o
    c1, c2 = sorted((table_cluster.get(table_of(pk), -1), table_cluster.get(table_of(fk), -1)))
    cands.append({"pk": pk, "fk": fk, "region": f"C{c1}<->C{c2}", "sim": sim(pk, fk),
                  "table_sim": table_sim(table_of(pk), table_of(fk))})
df_cands = pd.DataFrame(cands)

# --- 1. Cluster-pair ranking by pi(R) ---
if len(df_cands):
    regions = (df_cands.groupby("region")
               .agg(max_sim=("sim", "max"), max_tsim=("table_sim", "max"), n=("sim", "size"))
               .assign(pi=lambda d: 0.6 * d["max_sim"] + 0.4 * d["max_tsim"])
               .sort_values("pi", ascending=False))
else:
    regions = pd.DataFrame(columns=["max_sim", "max_tsim", "n", "pi"])

def pkfk_prompt(pk, fk) -> str:
    return (
        "Schema metadata only, no data values.\n"
        f"Candidate foreign key: {fk} ({dtype_map[fk]})\nCandidate referenced key: {pk} ({dtype_map[pk]})\n"
        f"Other columns of {table_of(fk)}: {', '.join(sorted(tables[table_of(fk)]['columns']))}\n"
        f"Other columns of {table_of(pk)}: {', '.join(sorted(tables[table_of(pk)]['columns']))}\n"
        "Decide whether the foreign key references the key, so that the two columns form a valid join.\n"
        'Answer as JSON: {"decision": "JOIN" or "NOT_JOIN", "probability": 0-1, "reason": "..."}'
    )

def prune_fk_targets(M_df, fk, accepted_pk) -> int:
    n = 0
    for other in key_attrs:
        if other != accepted_pk and other in M_df.index and M_df.loc[other, fk] == -1:
            set_pair(M_df, other, fk, 0.0); n += 1
    return n

capri_log, n_conflict_pruned, pending_regions = [], 0, []

for region, meta in regions.iterrows():
    sub = df_cands[df_cands["region"] == region].sort_values("sim", ascending=False)   # 2. candidate ranking
    batch_size = max(1, math.ceil(len(sub) * CAPRI_BATCH_RATIO))
    for start in range(0, len(sub), batch_size):
        batch = [r for r in sub.iloc[start:start + batch_size].itertuples(index=False)
                 if M.loc[r.pk, r.fk] == -1]              # already pruned/decided pairs are not re-verified
        if not batch:
            continue
        answers = llm_verify("capri_pkfk", [                # 3. batch verification
            {"item_id": f"{r.fk}|{r.pk}", "left_column": r.fk, "right_column": r.pk,
             "prompt": pkfk_prompt(r.pk, r.fk)} for r in batch])
        if len(answers) < len(batch):                       # manual mode: wait for results
            pending_regions.append(region); break
        any_accepted = False
        for r in batch:
            if M.loc[r.pk, r.fk] != -1:                     # pruned by an earlier acceptance in this batch
                continue
            ok, p = parse_decision(answers[f"{r.fk}|{r.pk}"], TAU_LLM_CAPRI)
            set_pair(M, r.pk, r.fk, p if ok else 0.0)
            capri_log.append({"region": region, "pk": r.pk, "fk": r.fk, "prob": p, "accepted": ok})
            if ok:
                any_accepted = True
                n_conflict_pruned += prune_fk_targets(M, r.fk, r.pk)   # 4. FK-target conflict pruning
        if not any_accepted:                                # 5. expand / terminate
            break

df_capri_log = pd.DataFrame(capri_log)
STAGE_CANDIDATES["after_capri"] = len(unknown_pairs(M))
print("Regions:", len(regions), "| CAPRI verified:", len(df_capri_log),
      "| accepted:", int(df_capri_log["accepted"].sum()) if len(df_capri_log) else 0,
      "| conflict-pruned:", n_conflict_pruned)
print("Unknown pairs handed to completion:", STAGE_CANDIDATES["after_capri"])

# Stage 4. Matrix completion + refinement

Unknown entries are estimated by iterative low-rank completion (truncated SVD) while observed
entries stay fixed. Pairs with completion score `< τ_prop` are pruned. Survivors that pass the
structural filter are verified by the LLM and accepted when `llm_score ≥ τ_llm`. Unverified
unknowns become `0` at finalization.

In [ ]:
from sklearn.decomposition import TruncatedSVD

def low_rank_complete(M_df: pd.DataFrame) -> pd.DataFrame:
    A = M_df.to_numpy(dtype=np.float32, copy=True)
    obs, miss = A != -1, A == -1
    A_obs = np.where(A > TAU_LLM, 1.0, A)                   # hard positives
    X = np.where(miss, float(A_obs[obs].mean()) if obs.any() else 0.0, A_obs).clip(0, 1)
    rank = min(COMPLETION_RANK, max(1, min(X.shape) - 1))
    for _ in range(COMPLETION_MAX_ITER):
        X_old = X.copy()
        svd = TruncatedSVD(n_components=rank, random_state=RANDOM_STATE)
        X_hat = svd.inverse_transform(svd.fit_transform(X))
        X = np.where(obs, A_obs, X_hat).clip(0, 1)
        X = (X + X.T) / 2; np.fill_diagonal(X, 0)
        if np.linalg.norm(X - X_old) < COMPLETION_TOL:
            break
    return pd.DataFrame(X, index=M_df.index, columns=M_df.columns)

def structurally_valid(a, b) -> bool:
    return (table_of(a) != table_of(b) and a not in not_key_attrs and b not in not_key_attrs
            and (a in key_attrs or b in key_attrs) and dtypes_compatible(a, b))

def refine_prompt(a, b, score) -> str:
    return (
        "Schema metadata only, no data values.\n"
        f"Column A: {a} ({dtype_map[a]})\nColumn B: {b} ({dtype_map[b]})\n"
        f"A matrix-completion model suggests these columns may join (score {score:.2f}).\n"
        "Accept only if they identify the same entity and A = B is a semantically valid join "
        "(including one column of a composite key). Do not accept only because names or datatypes look similar.\n"
        'Answer as JSON: {"decision": "JOIN" or "NOT_JOIN", "probability": 0-1, "reason": "..."}'
    )

completed = low_rank_complete(M)
refine_cands = []
for a, b in unknown_pairs(M):
    s = float(completed.loc[a, b])
    if s < TAU_PROP or not structurally_valid(a, b):
        set_pair(M, a, b, 0.0)
    else:
        refine_cands.append((a, b, s))

refine_answers = llm_verify("completion_refine", [
    {"item_id": f"{a}|{b}", "left_column": a, "right_column": b, "completion_score": s,
     "prompt": refine_prompt(a, b, s)} for a, b, s in refine_cands])

n_refine_accepted = 0
for a, b, s in refine_cands:
    ans = refine_answers.get(f"{a}|{b}")
    if ans is None:
        continue
    ok, p = parse_decision(ans, TAU_LLM)
    set_pair(M, a, b, p if ok else 0.0)
    n_refine_accepted += ok

print("Refinement candidates:", len(refine_cands), "| accepted:", n_refine_accepted)

# Stage 5. FK-guided PK–FK recovery

Some FK-like attributes are closer to existing FK members of a PK group than to the PK itself
(e.g. `cs_bill_customer_sk` vs. other customer FKs). For each still-unmatched, non-key, key-like
attribute, compare it with the accepted FK members of every PK group (same datatype family,
different table, not in the same logical group, `sim ≥ τ_sim`), keep the top-k per group, and ask the
LLM whether it belongs to the same group. On acceptance, the attribute is joined to the group's PK.

In [ ]:
def has_positive(M_df, a) -> bool:
    return bool((M_df.loc[a] > 0).any())

pk_groups = defaultdict(set)
for a, b in zip(*np.where(np.triu(M.values > 0, k=1))):
    o = orient(M.index[a], M.columns[b])
    if o:
        pk_groups[o[0]].add(o[1])

unmatched = [a for a in key_like if a not in key_attrs and not has_positive(M, a)]

fkfk_cands = []
for cand in unmatched:
    for pk, members in pk_groups.items():
        scored = []
        for fk in members:
            if (table_of(cand) == table_of(fk) or share_logical_group(cand, fk)
                    or share_logical_group(cand, pk) or not dtypes_compatible(cand, fk)):
                continue
            s = sim(cand, fk)
            if s >= TAU_SIM_FKFK:
                scored.append((s, fk))
        for s, fk in sorted(scored, reverse=True)[:FKFK_TOP_K_PER_GROUP]:
            fkfk_cands.append({"cand": cand, "fk": fk, "pk": pk, "sim": s})

def fkfk_prompt(cand, fk, pk) -> str:
    return (
        "Schema metadata only, no data values.\n"
        f"Known foreign key {fk} references {pk}.\nCandidate column: {cand} ({dtype_map[cand]})\n"
        f"Decide whether {cand} is another foreign key that references {pk}.\n"
        'Answer as JSON: {"decision": "SAME_GROUP" or "NOT_SAME_GROUP", "probability": 0-1, "reason": "..."}'
    )

fkfk_cands.sort(key=lambda r: -r["sim"])
fkfk_answers = llm_verify("fkfk_group", [
    {"item_id": f"{r['cand']}|{r['fk']}|{r['pk']}", "candidate_column": r["cand"],
     "existing_fk_member": r["fk"], "group_pk_column": r["pk"],
     "prompt": fkfk_prompt(r["cand"], r["fk"], r["pk"])} for r in fkfk_cands])

n_fkfk_assigned = 0
for r in fkfk_cands:
    if has_positive(M, r["cand"]):          # first strong assignment wins
        continue
    ans = fkfk_answers.get(f"{r['cand']}|{r['fk']}|{r['pk']}")
    if ans is None:
        continue
    ok, p = parse_decision(ans, TAU_LLM_FKFK)
    if ok:
        set_pair(M, r["pk"], r["cand"], p)
        n_fkfk_assigned += 1

print("Unmatched FK-like attributes:", len(unmatched), "| FK-FK checks:", len(fkfk_cands),
      "| recovered:", n_fkfk_assigned)

# Stage 6. Matrix finalization and evaluation

In [ ]:
M_final = M.copy()
M_final[M_final == -1] = 0.0
M_bin = (M_final >= TAU_LLM).astype(int)
for a in M_bin.index:
    M_bin.loc[a, a] = 0

pred_rows = []
for i, j in zip(*np.where(np.triu(M_bin.values == 1, k=1))):
    a, b = M_bin.index[i], M_bin.columns[j]
    kind = "PK-PK" if (a in key_attrs and b in key_attrs) else "PK-FK"
    pred_rows.append({"attr1": a, "attr2": b, "type": kind, "score": float(M_final.loc[a, b])})
df_pred = pd.DataFrame(pred_rows, columns=["attr1", "attr2", "type", "score"])
df_pred.to_csv(os.path.join(OUTPUT_DIR, "predicted_joins.csv"), index=False)

pred_set = {norm_pair(r.attr1, r.attr2) for r in df_pred.itertuples(index=False)}
tp, fp, fn = len(pred_set & gt_set), len(pred_set - gt_set), len(gt_set - pred_set)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

calls = Counter(e["task"] for e in LLM_CALL_LOG)
pair_level_calls = sum(v for k, v in calls.items() if k != "attribute_role")

print(f"=== {RUN_NAME} ===")
print(f"Predicted joins: {len(pred_set)}  (PK-PK {sum(df_pred['type']=='PK-PK')}, PK-FK {sum(df_pred['type']=='PK-FK')})")
print(f"P={precision:.2f}  R={recall:.2f}  F1={f1:.2f}  (TP={tp}, FP={fp}, FN={fn})")
print("Verifier calls by stage:", dict(calls))
print("Pair-level verifier calls:", pair_level_calls)
print("Candidate pairs by stage:", STAGE_CANDIDATES)

pd.DataFrame([{"dataset": RUN_NAME, "precision": precision, "recall": recall, "f1": f1,
               "pair_level_calls": pair_level_calls, **{f"calls_{k}": v for k, v in calls.items()}}]
            ).to_csv(os.path.join(OUTPUT_DIR, "summary.csv"), index=False)